In [1]:
import sys
import json
import random
import warnings
from copy import deepcopy
from concurrent.futures import ThreadPoolExecutor, as_completed

from tqdm import tqdm
import numpy as np
import pandas as pd
import openai
from openai import OpenAI, OpenAIError

import llm_connector.Collector as collector
import datasets.MBA as mba_ds

In [2]:
# Suppress the specific DtypeWarning
warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)

## 1. data pre-processing

In [3]:
ds_path = '../data/interim/funnel_mba_format.csv'
[mba_df, user_ids, user_num, user_ids_kv, item_names, item_num, items_kv, G_user, G_item] = mba_ds.MBA_load_data(ds_path)

Loading MBA dataset from path:../data/interim/funnel_mba_format.csv
all nan eliminated
totally 4297 unique users
totally 3846 unique items


388023it [00:06, 57707.85it/s]


In [4]:
# persona2idx_whole
with open('../data/persona2idx.json', 'r') as f:
    persona2idx_whole = json.load(f)

In [5]:
grouped_item_df = mba_df.groupby(['CustomerID', 'Itemname']).agg({'Quantity': 'sum'}).reset_index()

In [6]:
grouped_item_df

,CustomerID,Itemname,Quantity
0,12346,MEDIUM CERAMIC TOP STORAGE JAR,74215
1,12347,3D DOG PICTURE PLAYING CARDS,126
2,12347,3D SHEET OF CAT STICKERS,12
3,12347,3D SHEET OF DOG STICKERS,12
4,12347,60 TEATIME FAIRY CAKE CASES,72
...,...,...,...
263262,18287,STAR T-LIGHT HOLDER WILLIE WINKIE,12
263263,18287,STRAWBERRY BATH SPONGE,30
263264,18287,STRAWBERRY CERAMIC TRINKET BOX,12
263265,18287,SWISS CHALET TREE DECORATION,48


In [7]:
random.seed(191)

split_rate = [0.8, 0.2, 0.0] # train, validation, test
tri_graph_uidx2tidx_train, tri_graph_uidx2tidx_valid, tri_graph_uidx2tidx_test = {}, {}, {}
for uidx in G_user:
    items = list(G_user[uidx])
    random.shuffle(items)
    
    l = len(items)
    ltrain = max(int(split_rate[0]*l), 1)
    ltest = max(int(split_rate[2]*l), 0)
    
    tri_graph_uidx2tidx_train[uidx] = items[:ltrain]
    
    if ltrain < l:
        tri_graph_uidx2tidx_test[uidx] = items[ltrain:ltrain+ltest]
    else:
        tri_graph_uidx2tidx_test[uidx] = []

    if ltrain + ltest < l:
        tri_graph_uidx2tidx_valid[uidx] = items[ltrain+ltest:]
    else:
        tri_graph_uidx2tidx_valid[uidx] = []

## 2. pesona collecting

### 2.1 prompt eng

In [8]:
# make prompts, here we assign at most 8 personas on each user
prompt_sys = """Now you are an intelligent e-commerce domain assistant. You are skilled at summarizing, and capable of assigning high-level consumer personas based on a user's purchase behavior."""
prompt_user = """
Take a deep breath and work according to the instructions step by step.
Your goal is to identify users’ shopping behaviors based on products they have bought and label them with a given set of personas. You need to select at least one persona, at most 8 personas from our given persona list. But make sure that for each assignment you should find strong evidence in their purchase transactions. Please keep the procedure as accurate as possible.
Please provide the output in json format. Prefer to return arrays instead of comma separated strings. The following is an explanation of your return format:
{ 
    "user_number": ["Persona1", "Persona2", "Persona3"]
}

And here is an specific example:
{
    "12346": [ "Vegan/Vegetarian", "High-Protein Shopper", "Pet Owner"]
}

In the case that you feel there does not exist any suitable persona from the given list that can properly describe a user’s purchasing behavior, you can label the user as an ‘unrepresentable’ user as the following example:
{ 
    "12999": ["Unrepresentable"]
}

Here is the persona list you should choose from:
    1.  Vintage and Retro Enthusiast - A broad category that encompasses individuals with a love for vintage, retro, and nostalgia-themed items, including decorations, kitchenware, and accessories.
    2.  Gardening Lover - Those passionate about gardening activities, showcasing interests in both functional gardening tools and decorative garden items.
    3.  Crafting and DIY Hobbyist  - Individuals engaged in crafting, DIY projects, and creative activities, with purchases spanning across craft kits, sewing materials, and painting sets.
    4.  Home Decor Aficionado - Persons with a keen interest in enhancing their living spaces, evident in their choice of cozy, thematic, and design-focused home decor items.
    5.  Seasonal and Festive Decorator - This persona highlights an enthusiasm for decorating according to seasons and holidays, actively buying items for Christmas, Easter, and more.
    6.  Kitchen and Culinary Enthusiast - Individuals showing a strong passion for the culinary arts, from baking to cooking, with an emphasis on unique kitchen gadgets and themed kitchenware.
    7.  Tea and Coffee Lover - A persona defined by a love for tea and coffee rituals, as seen in their collection of mugs, tea sets, and related accessories.
    8.  Child and Family-centric Shopper - Reflects a focus on children or family, purchasing toys, educational items, and family-oriented products.
    9.  Comfort and Coziness Seeker - Individuals who prioritize items that provide comfort and warmth, including hot water bottles, hand warmers, and cozy textiles.
    10.  Animal and Pet Advocates - Shown by their preference for animal-themed decor and pet accessories, indicating a love for animals and nature.
    11.  Party and Event Planners - Persons who enjoy hosting and organizing parties and events, choosing bunting, decorations, and themed accessories to create joyful occasions.
    12.  Eco-friendly and Sustainable Goods Supporter - This persona prefers eco-friendly, sustainable products, showing an awareness of and commitment to environmental concerns.
    13.  Fashion Forward and Accessory Savvy - Individuals with an eye for fashion and accessories, ranging from trendy jewelry to stylish bags and fashion-forward decor.
    14.  Stationery and Organization Enthusiast - Represents those with a penchant for stationery, planners, and organizational tools, balancing aesthetics and functionality in their selections.
    15.  Outdoor and Adventure Prep - Persons interested in outdoor activities, including camping and picnics, as indicated by their choice of relevant gear and accessories.
    16.  Health and Wellness Focus - Those who purchase items related to personal health, wellness, and fitness, including yoga mats, health-conscious cookware, and wellness guides.
    17.  Educational and Learning Driven - Reflects an interest in self-improvement, educational toys, and games, showing a commitment to learning and intellectual development.
    18.  Baking and Confectionery Fans - Encompasses a love for baking, confectionery, and sweet treats, seen in purchases of baking sets, confectionery tools, and baking decor.
    19.  Art and Design Admirers - Individuals who appreciate art and design, buying items that reflect artistic value or are made by artisans, including wall art, sculptures, and design-centric decor.
    20.  Collector of Unique and Novelty Items - This persona looks for unique, whimsical, or novelty items, showcasing an appreciation for one-of-a-kind finds and interesting collectibles.

Remember that the user number (i.e., “user_number” in the example) should be exactly from the given transaction data, do not make it wrong since it is crucial.
"""

In [13]:
def assign_user_labels_train(
    client, # openai client object;
    grouped_item_df,    # pd.Dataframe;
    prompt_sys, prompt_user,    # str;
    uid,     # int;
    openai_model="gpt-4-turbo",   # openai model type
    debug=False, # bool;
) -> dict:

    # make transaction data for the training set
    uidx = user_ids_kv[uid]
    train_items = [item_names[t] for t in tri_graph_uidx2tidx_train[uidx]]
    transaction_data = collector.describe_user(uid, grouped_item_df, train_items)
    return transaction_data
    
    prompt_user_tail = f"""Here is the data of user {uid}'s transaction data for you to analyze:{transaction_data}
    Remind one more time that you can only select from the given 20 personas' list and only use the exactly given persona, you cannot use other words to describe.
    You do not need to explain how you get the result, so please respond no more than the required format.
    """

    if debug:
        print(prompt_user+prompt_user_tail)
        return

    try:
        completion = client.chat.completions.create(
            model=openai_model,
            messages=[
                {"role": "system", "content": prompt_sys},
                {"role": "user", "content": prompt_user+prompt_user_tail},
            ],
            stream=False,
        )

        response_result = ""
        # for chunk in stream:
        if completion.choices[0].message:
            response_result += completion.choices[0].message.content

        return {"user": uid, "users_profile": response_result}

    except Exception as e:  # Consider capturing a specific exception if possible
        print(f"[E] The following error occurred for user {uid} when collecting his persona labels: {e} ")
        return {"user": uid, "users_profile": "QUERY_FAILED"}

# unit test
auid = user_ids[55]
assign_user_labels_train(client=OpenAI(), grouped_item_df=grouped_item_df, prompt_sys=prompt_sys, prompt_user=prompt_user, uid=auid, debug=False)

'The user 12430 has totally purchased 8 unique products, we show each product name followed by its purchased times: he bought 3 PIECE SPACEBOY COOKIE CUTTER SET, 12 times; BOX OF VINTAGE ALPHABET BLOCKS, 4 times; MINI LIGHTS WOODLAND MUSHROOMS, 8 times; PARTY CHARMS 50 PIECES, 12 times; POSTAGE, 1 times; TRADTIONAL ALPHABET STAMP SET, 8 times; ZINC T-LIGHT HOLDER STAR LARGE, 12 times; ZINC T-LIGHT HOLDER STARS SMALL, 12 times'

### 2.2 label personas for mba dataset
⚠️ **Warning:** please note that this step may cause significant financial costs

In [ ]:
# we query with multiple python threads, since io-heavy tasks
client = OpenAI()

results = []
test_uids = deepcopy(user_ids)
sample_amount = len(test_uids)

with ThreadPoolExecutor(max_workers=12) as executor: # 5 workers: about 110s for 42 users; 10 workers: about 62s for 42 users
    future_results = [executor.submit(assign_user_labels_train, client, grouped_item_df, prompt_sys, prompt_user, uid,debug=False) for uid in test_uids]
    for future in tqdm(as_completed(future_results), total=sample_amount, desc="Processing Users"):
        result = future.result()
        results.append(result)

### 2.3 post-processing

In [ ]:
# defined_persona_set = ['Vintage and Retro Enthusiast',
 'Gardening Lover',
 'Crafting and DIY Hobbyist',
 'Home Decor Aficionado',
 'Seasonal and Festive Decorator',
 'Kitchen and Culinary Enthusiast',
 'Tea and Coffee Lover',
 'Child and Family-centric Shopper',
 'Comfort and Coziness Seeker',
 'Animal and Pet Advocates',
 'Party and Event Planners',
 'Eco-friendly and Sustainable Goods Supporter',
 'Fashion Forward and Accessory Savvy',
 'Stationery and Organization Enthusiast',
 'Outdoor and Adventure Prep',
 'Health and Wellness Focus',
 'Educational and Learning Driven',
 'Baking and Confectionery Fans',
 'Art and Design Admirers',
 'Collector of Unique and Novelty Items',
 'Unrepresentable']

persona_fix_map = {
    'Pet Owner': 'Animal and Pet Advocates',
    'Comfor and Coziness Seeker': 'Comfort and Coziness Seeker',
    'Coffee and Tea Lover': 'Tea and Coffee Lover',
    'Coffee Lover': 'Tea and Coffee Lover',
    'Kitchen and Culinary Enthusist': 'Kitchen and Culinary Enthusiast',
    'Collective of Unique and Novelty Items': 'Collector of Unique and Novelty Items',
    'Collection of Unique and Novelty Items': 'Collector of Unique and Novelty Items',
    'Collectors of Unique and Novelty Items': 'Collector of Unique and Novelty Items',
    'Collecter of Unique and Novelty Items': 'Collector of Unique and Novelty Items',
    'Jewelry Lover': 'Fashion Forward and Accessory Savvy',
    'Travel Enthusiast': 'Unrepresentable',
    'Cycling Enthusiast': 'Unrepresentable',
    'Patriotic Shopper': 'Unrepresentable',
    'Humor Enthusiast': 'Unrepresentable',
}

# checks
assert set([e['user'] for e in results]) == set(test_uids)

prep_results = []
for i in range(len(results)):
    record = results[i]
    uid = record['user']
    answer = record['users_profile']
    try:
        prep_results.append(collector.from_json_to_obj(str(uid), answer, defined_persona_set, persona_fix_map))
    except Exception as e:
        print(f'failed to process reccord {i}, with the following exception:')
        print(e)

In [ ]:
multi_labels = {}
for subdict in prep_results:
    multi_labels.update(subdict)
multi_labels = {int(k):v for k,v in multi_labels.items()}

tri_graph_uidx2pidx = {}
for uid, ps in multi_labels.items():
    # ignore the persona label of the unrepresentable users
    if 'Unrepresentable' in ps: continue
    uidx = user_ids_kv[uid]
    ps_idx = [persona2idx_whole[p] for p in ps]
    tri_graph_uidx2pidx[uidx] = ps_idx

In [35]:
# write the data into the file
# with open('../data/tri_graph_uidx2pidx.json', 'w') as f:
#     json.dump(tri_graph_uidx2pidx, f)
# with open('../data/tri_graph_uidx2tidx_train.json', 'w') as f:
#     json.dump(tri_graph_uidx2tidx_train, f)
# with open('../data/tri_graph_uidx2tidx_test.json', 'w') as f:
#     json.dump(tri_graph_uidx2tidx_valid, f)

## 3. associate personas with interested items

### 3.1 prompt engineering

In [36]:
predefined_persona_list = ['Vintage and Retro Enthusiast','Seasonal and Festive Decorator','Collector of Unique and Novelty Items','Home Decor Aficionado','Crafting and DIY Hobbyist','Baking and Confectionery Fans','Child and Family-centric Shopper','Gardening Lover','Comfort and Coziness Seeker','Kitchen and Culinary Enthusiast','Party and Event Planners','Tea and Coffee Lover','Stationery and Organization Enthusiast','Animal and Pet Advocates','Fashion Forward and Accessory Savvy','Educational and Learning Driven','Outdoor and Adventure Prep','Eco-friendly and Sustainable Goods Supporter','Art and Design Admirers','Health and Wellness Focus',]
system_prompt = """
Now you are an intelligent e-commerce domain assistant. You have a good understanding of various customer personas, and is skilled at connecting products with personas mostly likely to be interested in them.
"""
user_prompt = """
Take a deep breath and work according to the instructions step by step.

I will provide you a product name, and a set of customer personas summarized from an e-commerce dataset. The persona set includes 20 persona names and their corresponding definitions. 

Your task is to connect the product with a subset of personas in the persona set. Each of your selected persona should be likely to show a purchasing interest in this specific product, based on the product name and the persona definition.

Please adhere to the following principles when performing this task:
1. You should understand each persona strictly according to its definition, avoiding over-imaginations based on the persona's name or any hallucinations.
2. You can utilize your real-world knowledge to understand the attributes of the product as suggested by its name.
3. Ensure the connection made between the product and the persona must be well-substantiated. In other words, the product’s expected attributes should be highly relevant to the persona's definition.

Here are the 20 most representative personas:
    1.  Vintage and Retro Enthusiast - A broad category that encompasses individuals with a love for vintage, retro, and nostalgia-themed items, including decorations, kitchenware, and accessories.
    2.  Gardening Lover - Those passionate about gardening activities, showcasing interests in both functional gardening tools and decorative garden items.
    3.  Crafting and DIY Hobbyist  - Individuals engaged in crafting, DIY projects, and creative activities, with purchases spanning across craft kits, sewing materials, and painting sets.
    4.  Home Decor Aficionado - Persons with a keen interest in enhancing their living spaces, evident in their choice of cozy, thematic, and design-focused home decor items.
    5.  Seasonal and Festive Decorator - This persona highlights an enthusiasm for decorating according to seasons and holidays, actively buying items for Christmas, Easter, and more.
    6.  Kitchen and Culinary Enthusiast - Individuals showing a strong passion for the culinary arts, from baking to cooking, with an emphasis on unique kitchen gadgets and themed kitchenware.
    7.  Tea and Coffee Lover - A persona defined by a love for tea and coffee rituals, as seen in their collection of mugs, tea sets, and related accessories.
    8.  Child and Family-centric Shopper - Reflects a focus on children or family, purchasing toys, educational items, and family-oriented products.
    9.  Comfort and Coziness Seeker - Individuals who prioritize items that provide comfort and warmth, including hot water bottles, hand warmers, and cozy textiles.
    10.  Animal and Pet Advocates - Shown by their preference for animal-themed decor and pet accessories, indicating a love for animals and nature.
    11.  Party and Event Planners - Persons who enjoy hosting and organizing parties and events, choosing bunting, decorations, and themed accessories to create joyful occasions.
    12.  Eco-friendly and Sustainable Goods Supporter - This persona prefers eco-friendly, sustainable products, showing an awareness of and commitment to environmental concerns.
    13.  Fashion Forward and Accessory Savvy - Individuals with an eye for fashion and accessories, ranging from trendy jewelry to stylish bags and fashion-forward decor.
    14.  Stationery and Organization Enthusiast - Represents those with a penchant for stationery, planners, and organizational tools, balancing aesthetics and functionality in their selections.
    15.  Outdoor and Adventure Prep - Persons interested in outdoor activities, including camping and picnics, as indicated by their choice of relevant gear and accessories.
    16.  Health and Wellness Focus - Those who purchase items related to personal health, wellness, and fitness, including yoga mats, health-conscious cookware, and wellness guides.
    17.  Educational and Learning Driven - Reflects an interest in self-improvement, educational toys, and games, showing a commitment to learning and intellectual development.
    18.  Baking and Confectionery Fans - Encompasses a love for baking, confectionery, and sweet treats, seen in purchases of baking sets, confectionery tools, and baking decor.
    19.  Art and Design Admirers - Individuals who appreciate art and design, buying items that reflect artistic value or are made by artisans, including wall art, sculptures, and design-centric decor.
    20.  Collector of Unique and Novelty Items - This persona looks for unique, whimsical, or novelty items, showcasing an appreciation for one-of-a-kind finds and interesting collectibles.

To standardize the output, please follow the following rules:
1. You may connect a product to one to five of the most relevant personas. In fact, you should only list personas that are strongly related, even if there’re only two or three.
2. Reference following examples and generate your result in the same way, avoid any unnecessary explanations.

Example 1: for product named ‘TEA COSY RED STRIPE’, your response should be in the form of dictionary: “{"TEA COSY RED STRIPE": ["Tea and Coffee Lover", "Home Decor Aficionado", "Vintage and Retro Enthusiast"]}”.

Example 2: for ‘HAND WARMER BLUE RETROSPOT’, your response can be: “{"HAND WARMER BLUE RETROSPOT": ["Comfort and Coziness Seeker", "Vintage and Retro Enthusiast"]}”.

Example 3: {"CHERRY BLOSSOM PURSE": ["Fashion Forward and Accessory Savvy", "Collector of Unique and Novelty Items"]}

The name of the product I want you to analyze is:"""

In [46]:
# unit test
itemnames = [item_names[10]]
collector.query_all_items(OpenAI(), system_prompt, user_prompt, itemnames)

Total items: 1, start collecting...


Processing Items: 100%|████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.17s/it]


[{"POPPY'S PLAYHOUSE BEDROOM": '{"POPPY\'S PLAYHOUSE BEDROOM": ["Child and Family-centric Shopper"]}'}]

### 3.2 label personas for items
⚠️ **Warning:** please note that this step may cause significant financial costs

In [ ]:
item_labels = collector.query_all_items(item_names.tolist())

### 3.3 post-processing

In [ ]:
# process and generate the overall dictionary
item_labels_good = {}
for item in item_labels:
    item_labels_good.update(collector.parse_item_response(predefined_persona_list, item))

In [ ]:
tri_graph_tidx2pidx = {}
for name, ps in item_labels.items():
    assert not 'Unrepresentable' in ps
    tidx = items_kv[name]
    ps_idx = [persona2idx_whole[p] for p in ps]
    tri_graph_tidx2pidx[tidx] = ps_idx

In [50]:
# write the data into the file
# with open('../data/tri_graph_tidx2pidx.json', 'w') as f:
#     json.dump(tri_graph_tidx2pidx, f)